Baseline RAG Model and Inference Function

In [1]:
# import libraries for testing the setup
import sys
import torch

In [2]:
# Test the setup
print(sys.executable)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

/mnt/c/Users/jrhen/Desktop/NLP/NLP_Final_Project/ssm-rag/.venv_wsl/bin/python
2.14.1+cu126
12.6
True


In [3]:
# Baseline RAG Model

# Import libraries 
from transformers import AutoTokenizer
from huggingface_hub import login
from mamba_ssm import MambaLMHeadModel
from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel

/mnt/c/Users/jrhen/Desktop/NLP/NLP_Final_Project/ssm-rag/.venv_wsl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Initialize the model and tokenizer
model_id = "state-spaces/mamba3-siso-1.5b"
tokenizer_id = "meta-llama/Llama-3.1-8B"

In [5]:
# Run once per environment. Enter a Hugging Face token with read access.
login()

In [6]:
# Load thetokenizer
tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_id,
    token=True,
)

# Suppress the BPE clean_up warning
tokenizer.clean_up_tokenization_spaces = False

In [7]:
# Load the model
model = MambaLMHeadModel.from_pretrained(
    model_id,
    device="cuda",
    dtype=torch.bfloat16,
)

In [8]:
# Set the model to evaluation mode
print("Setting the model to evaluation mode...")
model.eval()

Setting the model to evaluation mode...


MambaLMHeadModel(
  (backbone): MixerModel(
    (embedding): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-23): 24 x Block(
        (norm): RMSNorm()
        (mixer): Mamba3(
          (in_proj): Linear(in_features=2048, out_features=8672, bias=False)
          (B_norm): RMSNorm()
          (C_norm): RMSNorm()
          (out_proj): Linear(in_features=4096, out_features=2048, bias=False)
        )
        (norm2): RMSNorm()
        (mlp): GatedMLP(
          (fc1): Linear(in_features=2048, out_features=8192, bias=False)
          (fc2): Linear(in_features=4096, out_features=2048, bias=False)
        )
      )
    )
    (norm_f): RMSNorm()
  )
  (lm_head): Linear(in_features=2048, out_features=128256, bias=False)
)

In [18]:
def generate_text(
    prompt: str,
    model,
    tokenizer,
    max_new_tokens: int = 64,
    temperature: float = 0.7,
    top_k: int = 50,
    top_p: float = 0.9,
    repetition_penalty: float = 1.1,
    do_sample: bool = True,
) -> str:
    """
    Run inference on the baseline Mamba-3 model.
    
    Args:
        prompt: Input text prompt.
        model: Loaded MambaLMHeadModel (already on CUDA, in eval mode).
        tokenizer: Llama-3.1-8B tokenizer.
        max_new_tokens: How many new tokens to generate.
        temperature: Sampling temperature (1.0 = no change, lower = more deterministic).
        top_k / top_p: Nucleus / top-k sampling.
        repetition_penalty: >1.0 discourages repeating the same tokens.
        do_sample: If False, falls back to greedy decoding (top_k=1).
    
    Returns:
        The full generated string (prompt + continuation).
    """
    # Tokenize and move to the same device as the model
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"].to(model.device if hasattr(model, "device") else "cuda")
    
    if max_new_tokens < 0:
        raise ValueError("max_new_tokens must be non-negative")
    if do_sample and temperature <= 0:
        raise ValueError("temperature must be greater than zero when sampling")
    if top_k < 0:
        raise ValueError("top_k must be non-negative")
    if not 0 <= top_p <= 1:
        raise ValueError("top_p must be between 0 and 1")
    if repetition_penalty <= 0:
        raise ValueError("repetition_penalty must be greater than zero")

    # mamba_ssm's generic generate() uses an inference step path that currently
    # passes 3-D hidden states to Mamba3.step(), which expects 2-D states.
    # Recompute the sequence at each step to stay on the supported forward path.
    with torch.inference_mode():
        for _ in range(max_new_tokens):
            logits = model(input_ids, num_last_tokens=1).logits[:, -1, :]
            if repetition_penalty != 1.0:
                token_scores = torch.gather(logits, 1, input_ids)
                token_scores = torch.where(
                    token_scores < 0,
                    token_scores * repetition_penalty,
                    token_scores / repetition_penalty,
                )
                logits.scatter_(1, input_ids, token_scores)
            if not do_sample:
                next_token = logits.argmax(dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k > 0:
                    top_k = min(top_k, logits.shape[-1])
                    threshold = torch.topk(logits, top_k, dim=-1).values[:, -1:]
                    logits = logits.masked_fill(logits < threshold, float("-inf"))
                if top_p > 0 and top_p < 1:
                    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                    cumulative_probs = torch.softmax(sorted_logits, dim=-1).cumsum(dim=-1)
                    sorted_remove = cumulative_probs > top_p
                    sorted_remove[:, 1:] = sorted_remove[:, :-1].clone()
                    sorted_remove[:, 0] = False
                    remove = torch.zeros_like(logits, dtype=torch.bool)
                    remove.scatter_(1, sorted_indices, sorted_remove)
                    logits = logits.masked_fill(remove, float("-inf"))
                next_token = torch.multinomial(torch.softmax(logits, dim=-1), num_samples=1)
            input_ids = torch.cat((input_ids, next_token), dim=1)
    
    output_ids = input_ids
    
    # Decode the whole sequence
    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return generated

In [19]:
prompt = "Mamba-3 is a state-space model that"

# Sampling (creative)
print("=== Sampling ===")
print(generate_text(prompt, model, tokenizer, max_new_tokens=80, temperature=0.8))

print("\n=== Greedy ===")
print(generate_text(prompt, model, tokenizer, max_new_tokens=80, do_sample=False))

=== Sampling ===
Mamba-3 is a state-space model that simulates the interaction of three predators (the Mamba, the Hunter and the Prey). It was originally developed for application to fish populations in Lake Victoria. This simulation system has been modified extensively by Michael Böhm.
The predator-prey relationship plays an important role as part of ecological interactions within lake systems. The basic food web structure follows:
As can be seen from this graph, there are

=== Greedy ===
Mamba-3 is a state-space model that describes the dynamics of an M-shaped robot. The main goal of this project was to develop and test a new method for modeling complex systems, which can be used in other applications as well.
The first step towards developing such models is to create a mathematical description of how each component interacts with its environment. This involves defining all possible states (positions) of every part along with their velocities at those positions;


In [16]:
def generate_simple(prompt, model, tokenizer, max_new_tokens=64, temperature=0.7, top_k=50):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.cuda()
    
    for _ in range(max_new_tokens):
        with torch.inference_mode():
            logits = model(input_ids).logits[:, -1, :]   # full forward every step
        
        if temperature == 0 or top_k == 1:
            next_token = logits.argmax(dim=-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k > 0:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        
        input_ids = torch.cat([input_ids, next_token], dim=1)
        
        if next_token.item() == tokenizer.eos_token_id:
            break
    
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

In [ ]:
print(generate_simple("Mamba-3 is a state-space model that", model, tokenizer, max_new_tokens=80))

Mamba-3 is a state-space model that allows the user to create a continuous-time, discrete-time, or hybrid model. It can be used to model nonlinear systems, time-varying systems, and differential equations. The Mamba-3 model is a MATLAB® toolbox that is used to create MATLAB® models. It can also be used to create Simulink® models and Simulink®-DirectX® models. You can


In [9]:
from typing import List, Optional, Callable
import numpy as np

In [10]:
# ------------------------------------------------------------
# Helper: simple chunking (you can replace with any splitter)
# ------------------------------------------------------------
def split_corpus(token_ids: torch.Tensor, chunk_size: int = 512) -> List[torch.Tensor]:
    """Split a 1-D token tensor into fixed-size chunks."""
    chunks = []
    for i in range(0, len(token_ids), chunk_size):
        chunks.append(token_ids[i : i + chunk_size])
    return chunks

In [11]:
# ------------------------------------------------------------
# Core inference / indexing function following the diagram
# ------------------------------------------------------------
@torch.inference_mode()
def index_corpus_with_mamba(
    corpus: str,
    model,
    tokenizer,
    vector_db,                          # any object with .add(ids, vectors)
    chunk_size: int = 512,
    device: str = "cuda",
) -> None:
    """
    Follows the flowchart:

        Start → Tokenize → Split Corpus
                              ↓
                         Next Chunk
                              ↓
                        Update State  →  Extract state to Vector DB
                              ↑                    ↓
                              └──── Corpus Empty? ──┘
                                         │ Yes
                                         ↓
                                        End
    """
    # 1. Tokenize
    token_ids = tokenizer(
        corpus,
        return_tensors="pt",
        add_special_tokens=False,
    ).input_ids[0].to(device)          # shape: (seq_len,)

    # 2. Split Corpus
    chunks = split_corpus(token_ids, chunk_size=chunk_size)

    # 3. Loop: while corpus is not empty
    for chunk_idx, chunk in enumerate(chunks):
        # --- Update State ---
        # Feed the chunk through the model.  
        # We only need the final hidden state / SSM state.
        # For MambaLMHeadModel the cleanest way is a full forward
        # and take the last-layer hidden state (or the SSM state if exposed).

        input_ids = chunk.unsqueeze(0)          # (1, chunk_len)

        # Forward pass – this updates the internal recurrent state
        outputs = model(input_ids)
        # outputs.logits shape: (1, chunk_len, vocab)

        # --- Extract state to Vector DB ---
        # Option A (simplest & most common for baseline RAG):
        # use the last token's hidden state as the chunk embedding.
        # (If your model exposes intermediate states you can replace this.)
        with torch.no_grad():
            # Re-run just to get hidden states if the model supports it
            # Many MambaLMHeadModel versions return only logits.
            # A practical baseline is to average the token embeddings
            # or take the final logit-derived representation.
            # Here we use the mean of the last-layer features via a hook
            # or simply the mean of the input embeddings as a fallback.

            # ---- Practical extraction for current mamba_ssm ----
            # Grab the embedding of the chunk and mean-pool
            emb = model.backbone.embedding(input_ids)          # (1, L, d)
            state_vector = emb.mean(dim=1).squeeze(0)          # (d,)
            state_vector = state_vector.float().cpu().numpy()

        # Store in vector DB
        vector_db.add(
            ids=[f"chunk_{chunk_idx}"],
            embeddings=[state_vector],
            metadatas=[{"text": tokenizer.decode(chunk, skip_special_tokens=True)}],
        )

        # (The loop continues automatically until chunks are exhausted
        #  ≡ “Corpus Empty? → Yes → End”)

In [12]:
# ------------------------------------------------------------
# Minimal in-memory Vector DB (replace with Chroma / FAISS / etc.)
# ------------------------------------------------------------
class SimpleVectorDB:
    def __init__(self):
        self.ids = []
        self.embeddings = []
        self.metadatas = []

    def add(self, ids, embeddings, metadatas=None):
        self.ids.extend(ids)
        self.embeddings.extend(embeddings)
        if metadatas:
            self.metadatas.extend(metadatas)

    def search(self, query_vector, top_k=5):
        if not self.embeddings:
            return []
        embs = np.stack(self.embeddings)
        scores = embs @ query_vector
        idx = np.argsort(scores)[::-1][:top_k]
        return [(self.ids[i], float(scores[i]), self.metadatas[i]) for i in idx]

In [13]:
# ------------------------------------------------------------
# Example usage
# ------------------------------------------------------------
vector_db = SimpleVectorDB()

corpus = """Your long document or collection of documents goes here..."""

index_corpus_with_mamba(
    corpus=corpus,
    model=model,
    tokenizer=tokenizer,
    vector_db=vector_db,
    chunk_size=512,
)

print(f"Indexed {len(vector_db.ids)} chunks")

Indexed 1 chunks
